In [3]:
!pip -q uninstall -y transformers peft trl accelerate bitsandbytes
!pip -q install -U \
  "transformers==5.5.4" \
  "peft>=0.17.0" \
  "trl>=0.19.0" \
  "accelerate>=1.10.0" \
  "bitsandbytes>=0.47.0" \
  "datasets>=4.1.1" \
  "sentencepiece" \
  "scikit-learn" \
  "pyyaml"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 133.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 158.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 58.1 MB/s eta 0:00:00


In [4]:
import os
import gc
import json
import math
import random
import inspect
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import Counter, defaultdict

import yaml
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split

import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    set_seed,
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model,
    PeftModel,
)

In [5]:
print("transformers version:", transformers.__version__)
print("TrainingArguments module:", TrainingArguments.__module__)
print("TrainingArguments file:", inspect.getfile(TrainingArguments))
print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

transformers version: 5.5.4
TrainingArguments module: transformers.training_args
TrainingArguments file: /usr/local/lib/python3.12/dist-packages/transformers/training_args.py
torch version: 2.10.0+cu128
cuda available: True
gpu: NVIDIA L4


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
@dataclass
class Config:
    # paths
    DATA_PATH: str = "/content/drive/MyDrive/Data298ModelTraining/k8s_combined_incidents.jsonl"
    OUTPUT_DIR: str = "/content/drive/MyDrive/Data298ModelTraining/checkpoints/deepseek_rca_7b"
    LABEL_MAP_PATH: str = "/content/drive/MyDrive/Data298ModelTraining/scenario_rca.yaml"

    # model
    MODEL_ID: str = "deepseek-ai/deepseek-llm-7b-chat"

    # seed
    SEED: int = 42

    # split
    TEST_SIZE: float = 0.10
    VAL_SIZE: float = 0.10

    # sequence
    MAX_SEQ_LEN: int = 2048

    # training
    EPOCHS: int = 3
    LEARNING_RATE: float = 1e-4
    WEIGHT_DECAY: float = 0.01
    WARMUP_RATIO: float = 0.03
    PER_DEVICE_TRAIN_BATCH_SIZE: int = 1
    PER_DEVICE_EVAL_BATCH_SIZE: int = 1
    GRADIENT_ACCUMULATION_STEPS: int = 8

    # lora
    LORA_R: int = 16
    LORA_ALPHA: int = 32
    LORA_DROPOUT: float = 0.05

    # log/save/eval
    LOGGING_STEPS: int = 10
    SAVE_STEPS: int = 100
    EVAL_STEPS: int = 100
    SAVE_TOTAL_LIMIT: int = 3

    # quantization
    LOAD_IN_4BIT: bool = True

    # resume
    RESUME_IF_CHECKPOINT_EXISTS: bool = True

cfg = Config()
set_seed(cfg.SEED)
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

print(asdict(cfg))

{'DATA_PATH': '/content/drive/MyDrive/Data298ModelTraining/k8s_combined_incidents.jsonl', 'OUTPUT_DIR': '/content/drive/MyDrive/Data298ModelTraining/checkpoints/deepseek_rca_7b', 'LABEL_MAP_PATH': '/content/drive/MyDrive/Data298ModelTraining/scenario_rca.yaml', 'MODEL_ID': 'deepseek-ai/deepseek-llm-7b-chat', 'SEED': 42, 'TEST_SIZE': 0.1, 'VAL_SIZE': 0.1, 'MAX_SEQ_LEN': 2048, 'EPOCHS': 3, 'LEARNING_RATE': 0.0001, 'WEIGHT_DECAY': 0.01, 'WARMUP_RATIO': 0.03, 'PER_DEVICE_TRAIN_BATCH_SIZE': 1, 'PER_DEVICE_EVAL_BATCH_SIZE': 1, 'GRADIENT_ACCUMULATION_STEPS': 8, 'LORA_R': 16, 'LORA_ALPHA': 32, 'LORA_DROPOUT': 0.05, 'LOGGING_STEPS': 10, 'SAVE_STEPS': 100, 'EVAL_STEPS': 100, 'SAVE_TOTAL_LIMIT': 3, 'LOAD_IN_4BIT': True, 'RESUME_IF_CHECKPOINT_EXISTS': True}


In [8]:
data_path = Path(cfg.DATA_PATH)
print("Dataset path:", data_path)
print("Exists:", data_path.exists())

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {cfg.DATA_PATH}")

Dataset path: /content/drive/MyDrive/Data298ModelTraining/k8s_combined_incidents.jsonl
Exists: True


In [9]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON at line {i}: {e}")
    return rows

rows = load_jsonl(cfg.DATA_PATH)
print(f"Loaded rows: {len(rows):,}")
print("Sample keys:", sorted(rows[0].keys()))
print("First sample preview:")
print(json.dumps(rows[0], indent=2)[:2000])

Loaded rows: 7,000
Sample keys: ['claim_name', 'container_state', 'event_message', 'event_reason', 'evidence_text', 'image', 'last_state', 'namespace', 'node', 'node_selectors', 'pod_describe', 'pod_logs', 'pod_logs_previous', 'pod_name', 'pod_status', 'ready', 'restart_count', 'scenario_id', 'service_account_name']
First sample preview:
{
  "scenario_id": "pvc_not_found_mountfail",
  "namespace": "data-pipeline",
  "pod_name": "busybox-pn3zirfh-86bbb7957c-7754l",
  "service_account_name": "default",
  "node": "<none>",
  "pod_status": "Pending",
  "image": "busybox:1.36",
  "container_state": null,
  "last_state": null,
  "ready": null,
  "restart_count": null,
  "node_selectors": "<none>",
  "claim_name": "missing-pvc-pn3zirfh",
  "event_reason": "FailedScheduling",
  "event_message": "0/1 nodes are available: persistentvolumeclaim \"missing-pvc-pn3zirfh\" not found. preemption: 0/1 nodes are available: 1 Preemption is not helpful for scheduling.",
  "pod_describe": "Name:           

In [10]:
scenario_counts = Counter(r.get("scenario_id", "MISSING") for r in rows)
print("Scenario counts:")
for k, v in sorted(scenario_counts.items()):
    print(f"{k:45s} {v}")

Scenario counts:
crashloop_bad_args                            500
createcontainerconfigerror_bad_configmap_key  500
createcontainerconfigerror_missing_secret     500
dns_resolution_failure                        500
failedscheduling_insufficient_cpu             500
failedscheduling_insufficient_memory          500
failedscheduling_taint                        500
imagepull_bad_tag                             500
imagepull_registry_auth                       500
nodeselector_mismatch                         500
oomkilled_limit_too_low                       500
pvc_not_found_mountfail                       512
rbac_forbidden                                488
service_connection_refused                    500


In [11]:
DEFAULT_SCENARIO_RCA = {
    "pvc_not_found_mountfail": (
        "Root cause: The workload fails during volume setup because the referenced PersistentVolumeClaim does not exist or cannot be resolved in the namespace.\n"
        "Why: Kubernetes cannot mount a volume from a claim that is missing or not available to the pod.\n"
        "Remediation:\n"
        "- Verify the PVC name and namespace.\n"
        "- Create the missing PVC or fix the claim reference.\n"
        "- Confirm the PVC is bound before redeploying."
    ),
    "failedscheduling_taint": (
        "Root cause: The pod is unschedulable because nodes have taints that the pod does not tolerate.\n"
        "Why: The scheduler rejects nodes whose taints are not matched by pod tolerations.\n"
        "Remediation:\n"
        "- Inspect node taints.\n"
        "- Add required tolerations if appropriate.\n"
        "- Or target compatible nodes."
    ),
    "dns_resolution_failure": (
        "Root cause: The workload cannot resolve the target service or hostname because DNS resolution is failing.\n"
        "Why: Pod name resolution depends on cluster DNS and correct service naming.\n"
        "Remediation:\n"
        "- Verify service name, namespace, and DNS suffix.\n"
        "- Check CoreDNS health and logs.\n"
        "- Validate pod DNS policy and connectivity."
    ),
    "service_connection_refused": (
        "Root cause: The application reaches the endpoint but the connection is refused because the service is not listening or not reachable on the expected port.\n"
        "Why: A refused connection usually means no process is accepting traffic on that target IP:port.\n"
        "Remediation:\n"
        "- Verify the container is listening on the expected port.\n"
        "- Check Service targetPort and Endpoints.\n"
        "- Confirm readiness and startup state."
    ),
    "createcontainerconfigerror_missing_secret": (
        "Root cause: The pod cannot start because it references a missing Secret required by the container configuration.\n"
        "Why: Kubernetes blocks container startup when a required Secret reference cannot be resolved.\n"
        "Remediation:\n"
        "- Verify the Secret name and namespace.\n"
        "- Create the missing Secret.\n"
        "- Check env, envFrom, and volume references."
    ),
    "createcontainerconfigerror_bad_configmap_key": (
        "Root cause: The pod cannot start because the ConfigMap reference uses a missing or invalid key.\n"
        "Why: Kubernetes cannot inject data from a ConfigMap key that does not exist.\n"
        "Remediation:\n"
        "- Verify the ConfigMap exists.\n"
        "- Confirm the referenced key is present.\n"
        "- Correct the manifest and redeploy."
    ),
    "imagepull_bad_tag": (
        "Root cause: The image cannot be pulled because the specified image tag does not exist.\n"
        "Why: The registry cannot return an image manifest for a tag that is missing.\n"
        "Remediation:\n"
        "- Verify the repository and tag.\n"
        "- Push the expected image if needed.\n"
        "- Update the workload to a valid tag."
    ),
    "crashloop_bad_args": (
        "Root cause: The container repeatedly crashes because the startup command or arguments are invalid for the application.\n"
        "Why: The process exits during startup, so Kubernetes keeps restarting it and enters CrashLoopBackOff.\n"
        "Remediation:\n"
        "- Inspect container logs.\n"
        "- Verify command and args.\n"
        "- Correct the startup configuration."
    ),
    "oomkilled_limit_too_low": (
        "Root cause: The container is OOMKilled because its memory limit is too low for the workload.\n"
        "Why: The kernel terminates the container when it exceeds its cgroup memory limit.\n"
        "Remediation:\n"
        "- Increase the memory limit.\n"
        "- Check actual memory usage.\n"
        "- Tune application memory usage if needed."
    ),
    "imagepull_registry_auth": (
        "Root cause: The image pull fails because registry authentication is missing or invalid.\n"
        "Why: The registry denies access when credentials or imagePullSecrets are absent or incorrect.\n"
        "Remediation:\n"
        "- Verify imagePullSecrets.\n"
        "- Check registry credentials.\n"
        "- Confirm the Secret is attached to the workload or service account."
    ),
    "failedscheduling_insufficient_memory": (
        "Root cause: The pod is unschedulable because no node has enough allocatable memory for the request.\n"
        "Why: The scheduler cannot place the pod if requested memory exceeds available capacity on all candidate nodes.\n"
        "Remediation:\n"
        "- Reduce memory requests if appropriate.\n"
        "- Add capacity or scale the node pool.\n"
        "- Check for resource fragmentation."
    ),
    "failedscheduling_insufficient_cpu": (
        "Root cause: The pod is unschedulable because no node has enough allocatable CPU for the request.\n"
        "Why: The scheduler cannot bind the pod when requested CPU is not available on any node.\n"
        "Remediation:\n"
        "- Reduce CPU requests if appropriate.\n"
        "- Add capacity or scale the node pool.\n"
        "- Review cluster utilization."
    ),
    "nodeselector_mismatch": (
        "Root cause: The pod cannot be scheduled because its nodeSelector does not match available node labels.\n"
        "Why: The scheduler filters out nodes that do not satisfy the selector constraints.\n"
        "Remediation:\n"
        "- Inspect node labels.\n"
        "- Fix the nodeSelector.\n"
        "- Or add matching labels to intended nodes."
    ),
    "rbac_forbidden": (
        "Root cause: The workload is denied by Kubernetes RBAC because its identity lacks the required permissions.\n"
        "Why: The API server rejects requests that are not allowed by the bound roles for the service account.\n"
        "Remediation:\n"
        "- Identify the service account in use.\n"
        "- Review Role or ClusterRole bindings.\n"
        "- Grant only the minimum required permissions."
    ),
}

In [12]:
def load_scenario_rca(label_map_path, default_map):
    merged = dict(default_map)
    label_path = Path(label_map_path)

    if label_path.exists():
        print(f"Loading RCA mapping from: {label_path}")
        with open(label_path, "r", encoding="utf-8") as f:
            if label_path.suffix.lower() in {".yaml", ".yml"}:
                loaded = yaml.safe_load(f)
            else:
                loaded = json.load(f)

        if loaded is None:
            loaded = {}

        if not isinstance(loaded, dict):
            raise ValueError("scenario_rca file must contain a dict: scenario_id -> RCA text")

        merged.update(loaded)
    else:
        print(f"No label map found. Writing default template to: {label_path}")
        label_path.parent.mkdir(parents=True, exist_ok=True)
        with open(label_path, "w", encoding="utf-8") as f:
            yaml.safe_dump(default_map, f, sort_keys=True, allow_unicode=True)

    return merged

SCENARIO_RCA = load_scenario_rca(cfg.LABEL_MAP_PATH, DEFAULT_SCENARIO_RCA)
print("Loaded RCA labels:", len(SCENARIO_RCA))

Loading RCA mapping from: /content/drive/MyDrive/Data298ModelTraining/scenario_rca.yaml
Loaded RCA labels: 14


In [13]:
CANDIDATE_TARGET_FIELDS = [
    "target",
    "output",
    "response",
    "answer",
    "rca",
    "rca_text",
    "label_text",
    "ground_truth",
]

def get_target_text(row):
    for key in CANDIDATE_TARGET_FIELDS:
        val = row.get(key, None)
        if isinstance(val, str) and val.strip():
            return val.strip()

    scenario_id = row.get("scenario_id")
    if scenario_id in SCENARIO_RCA:
        return SCENARIO_RCA[scenario_id].strip()

    raise KeyError(
        "No target text found. Either add one of these fields "
        f"{CANDIDATE_TARGET_FIELDS} to each row, or provide scenario_id mapping."
    )

print("Example target:\n")
print(get_target_text(rows[0]))

Example target:

The workload fails during volume setup because the referenced PersistentVolumeClaim does not exist or cannot be resolved in the namespace. Kubernetes cannot mount the requested storage, so the pod never starts normally.
Remediation:
- Verify the PVC name and namespace.
- Create the missing PVC or fix the claim reference in the workload spec.
- Confirm the bound volume is available before redeploying.


In [14]:

def compact_evidence(row):
    evidence = (row.get("evidence_text") or "").strip()
    namespace = row.get("namespace", "")
    pod_name = row.get("pod_name", "")
    pod_status = row.get("pod_status", "")
    event_reason = row.get("event_reason", "")
    event_message = row.get("event_message", "")

    parts = [
        "Task: Analyze the Kubernetes incident and produce RCA.",
        "",
        "Return exactly this format:",
        "Root cause: ...",
        "Why: ...",
        "Remediation:",
        "- ...",
        "- ...",
        "",
        "Incident evidence:",
        f"Namespace: {namespace}",
        f"Pod: {pod_name}",
        f"Pod status: {pod_status}",
        f"Event reason: {event_reason}",
        f"Event message: {event_message}",
        "",
        "Evidence:",
        evidence,
    ]
    return "\n".join(parts).strip()

In [15]:
tokenizer = AutoTokenizer.from_pretrained(
    cfg.MODEL_ID,
    trust_remote_code=True,
    use_fast=False,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Model:", cfg.MODEL_ID)
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)
print("Padding side:", tokenizer.padding_side)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/594 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model: deepseek-ai/deepseek-llm-7b-chat
Pad token: <｜end▁of▁sentence｜>
EOS token: <｜end▁of▁sentence｜>
Padding side: right


In [16]:
def build_messages(row):
    return [
        {
            "role": "user",
            "content": compact_evidence(row)
        }
    ]

def build_prompt(row):
    messages = build_messages(row)
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    return prompt

print(build_prompt(rows[0])[:2000])

<｜begin▁of▁sentence｜>User: Task: Analyze the Kubernetes incident and produce RCA.

Return exactly this format:
Root cause: ...
Why: ...
Remediation:
- ...
- ...

Incident evidence:
Namespace: data-pipeline
Pod: busybox-pn3zirfh-86bbb7957c-7754l
Pod status: Pending
Event reason: FailedScheduling
Event message: 0/1 nodes are available: persistentvolumeclaim "missing-pvc-pn3zirfh" not found. preemption: 0/1 nodes are available: 1 Preemption is not helpful for scheduling.

Evidence:
POD DESCRIBE:
Name:             busybox-pn3zirfh-86bbb7957c-7754l
Namespace:        data-pipeline
Priority:         0
Service Account:  default
Node:             <none>
Labels:           app=busybox-pn3zirfh
                  pod-template-hash=86bbb7957c
                  run_id=pn3zirfh
                  scenario_id=pvc_not_found_mountfail
                  synthetic=true
Annotations:      <none>
Status:           Pending
IP:
IPs:              <none>
Controlled By:    ReplicaSet/busybox-pn3zirfh-86bbb7957c
Con

In [17]:
indices = list(range(len(rows)))
labels = [r["scenario_id"] for r in rows]

train_idx, test_idx = train_test_split(
    indices,
    test_size=cfg.TEST_SIZE,
    random_state=cfg.SEED,
    stratify=labels,
)

train_labels = [labels[i] for i in train_idx]
val_relative = cfg.VAL_SIZE / (1.0 - cfg.TEST_SIZE)

train_idx, val_idx = train_test_split(
    train_idx,
    test_size=val_relative,
    random_state=cfg.SEED,
    stratify=train_labels,
)

train_rows = [rows[i] for i in train_idx]
val_rows = [rows[i] for i in val_idx]
test_rows = [rows[i] for i in test_idx]

print(f"Train: {len(train_rows)}")
print(f"Val:   {len(val_rows)}")
print(f"Test:  {len(test_rows)}")

Train: 5600
Val:   700
Test:  700


In [18]:
def to_record(row):
    return {
        "scenario_id": row["scenario_id"],
        "prompt": build_prompt(row),
        "completion": get_target_text(row),
        "raw_evidence": row.get("evidence_text", ""),
    }

train_ds = Dataset.from_list([to_record(r) for r in train_rows])
val_ds = Dataset.from_list([to_record(r) for r in val_rows])
test_ds = Dataset.from_list([to_record(r) for r in test_rows])

print(train_ds)
print(val_ds)
print(test_ds)
print(train_ds[0])

Dataset({
    features: ['scenario_id', 'prompt', 'completion', 'raw_evidence'],
    num_rows: 5600
})
Dataset({
    features: ['scenario_id', 'prompt', 'completion', 'raw_evidence'],
    num_rows: 700
})
Dataset({
    features: ['scenario_id', 'prompt', 'completion', 'raw_evidence'],
    num_rows: 700
})
{'scenario_id': 'createcontainerconfigerror_bad_configmap_key', 'prompt': '<｜begin▁of▁sentence｜>User: Task: Analyze the Kubernetes incident and produce RCA.\n\nReturn exactly this format:\nRoot cause: ...\nWhy: ...\nRemediation:\n- ...\n- ...\n\nIncident evidence:\nNamespace: team-a-dev-v8rq8p\nPod: web-w-9cj8-0\nPod status: Pending\nEvent reason: Scheduled\nEvent message: Successfully assigned team-a-dev-v8rq8p/web-w-9cj8-0 to k8s-incidents-control-plane\n\nEvidence:\nnamespace: team-a-dev-v8rq8p\nworkload: web-w-9cj8\ncontainer: sidecar\nimage: docker.io/library/nginx:2.0.1\n=== kubectl get pods ===\nNAME           READY   STATUS         RESTARTS   AGE\nweb-w-9cj8-0   0/1     ErrIma

In [19]:
def tokenize_example(example):
    prompt_ids = tokenizer(
        example["prompt"],
        add_special_tokens=False,
        truncation=True,
        max_length=cfg.MAX_SEQ_LEN,
    )["input_ids"]

    completion_ids = tokenizer(
        example["completion"],
        add_special_tokens=False,
        truncation=True,
        max_length=cfg.MAX_SEQ_LEN,
    )["input_ids"]

    # reserve 1 token for EOS
    max_completion_len = cfg.MAX_SEQ_LEN - len(prompt_ids) - 1

    if max_completion_len < 1:
        # keep some space for completion if prompt is too long
        prompt_ids = prompt_ids[: cfg.MAX_SEQ_LEN - 256]
        max_completion_len = cfg.MAX_SEQ_LEN - len(prompt_ids) - 1

    completion_ids = completion_ids[:max_completion_len]

    input_ids = prompt_ids + completion_ids + [tokenizer.eos_token_id]
    attention_mask = [1] * len(input_ids)
    labels = ([-100] * len(prompt_ids)) + completion_ids + [tokenizer.eos_token_id]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "length": len(input_ids),
        "scenario_id": example["scenario_id"],
    }

train_tok = train_ds.map(tokenize_example, remove_columns=train_ds.column_names)
val_tok = val_ds.map(tokenize_example, remove_columns=val_ds.column_names)

print(train_tok[0].keys())
print("Example token length:", len(train_tok[0]["input_ids"]))

Map:   0%|          | 0/5600 [00:00<?, ? examples/s]

Map:   0%|          | 0/700 [00:00<?, ? examples/s]

dict_keys(['scenario_id', 'input_ids', 'attention_mask', 'labels', 'length'])
Example token length: 1305


In [20]:
train_lengths = [train_tok[i]["length"] for i in range(min(500, len(train_tok)))]
print("Length stats on first 500 train samples:")
print("min:", min(train_lengths))
print("max:", max(train_lengths))
print("avg:", sum(train_lengths) / len(train_lengths))
print("MAX_SEQ_LEN:", cfg.MAX_SEQ_LEN)

Length stats on first 500 train samples:
min: 568
max: 1876
avg: 1171.758
MAX_SEQ_LEN: 2048


In [21]:
class SupervisedCollator:
    def __init__(self, tokenizer):
        self.pad_token_id = tokenizer.pad_token_id

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)

        input_ids = []
        attention_mask = []
        labels = []

        for f in features:
            pad_len = max_len - len(f["input_ids"])
            input_ids.append(f["input_ids"] + [self.pad_token_id] * pad_len)
            attention_mask.append(f["attention_mask"] + [0] * pad_len)
            labels.append(f["labels"] + [-100] * pad_len)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

data_collator = SupervisedCollator(tokenizer)
print("Collator ready")

Collator ready


In [22]:
from peft import PeftModel, LoraConfig, prepare_model_for_kbit_training, get_peft_model
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for this training setup.")

if torch.cuda.is_bf16_supported():
    compute_dtype = torch.bfloat16
    use_bf16 = True
    use_fp16 = False
else:
    compute_dtype = torch.float16
    use_bf16 = False
    use_fp16 = True

bnb_config = BitsAndBytesConfig(
    load_in_4bit=cfg.LOAD_IN_4BIT,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

base_model = AutoModelForCausalLM.from_pretrained(
    cfg.MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model)

if latest_checkpoint is not None:
    model = PeftModel.from_pretrained(
        base_model,
        latest_checkpoint,
        is_trainable=True,
    )
    print("Loaded model weights from checkpoint:", latest_checkpoint)
else:
    peft_config = LoraConfig(
        r=cfg.LORA_R,
        lora_alpha=cfg.LORA_ALPHA,
        lora_dropout=cfg.LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "up_proj", "down_proj", "gate_proj"
        ],
    )
    model = get_peft_model(base_model, peft_config)
    print("Started fresh training")

model.gradient_checkpointing_enable()
model.print_trainable_parameters()

pytorch_model.bin.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

NameError: name 'latest_checkpoint' is not defined

In [ ]:
training_args = TrainingArguments(
    output_dir=cfg.OUTPUT_DIR + "_newrun",
    num_train_epochs=cfg.EPOCHS,
    per_device_train_batch_size=cfg.PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=cfg.PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=cfg.GRADIENT_ACCUMULATION_STEPS,
    learning_rate=cfg.LEARNING_RATE,
    weight_decay=cfg.WEIGHT_DECAY,
    warmup_ratio=cfg.WARMUP_RATIO,
    logging_steps=cfg.LOGGING_STEPS,
    save_steps=cfg.SAVE_STEPS,
    eval_steps=cfg.EVAL_STEPS,
    eval_strategy="steps",
    save_strategy="steps",
    save_total_limit=cfg.SAVE_TOTAL_LIMIT,
    bf16=use_bf16,
    fp16=use_fp16,
    report_to="none",
    lr_scheduler_type="cosine",
    seed=cfg.SEED,
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    optim="adamw_torch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    train_sampling_strategy="group_by_length",
    length_column_name="length",
)

print(training_args)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("Trainer created successfully")

Trainer created successfully


In [ ]:
train_result = trainer.train()

trainer.save_model(training_args.output_dir)
tokenizer.save_pretrained(training_args.output_dir)

print(train_result)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 100001}.


Step,Training Loss,Validation Loss
100,0.059965,0.102489
200,0.014852,0.071277
300,0.005438,0.016867
400,0.004002,0.007286
500,0.000287,0.000325
600,0.000109,0.000222
700,0.000083,0.000083
800,0.000056,0.000059
900,0.000046,0.000051


In [23]:
from pathlib import Path

def get_latest_checkpoint(output_dir):
    output_path = Path(output_dir)
    if not output_path.exists():
        return None

    checkpoints = []
    for p in output_path.iterdir():
        if p.is_dir() and p.name.startswith("checkpoint-"):
            try:
                step = int(p.name.split("-")[-1])
                checkpoints.append((step, str(p)))
            except ValueError:
                pass

    if not checkpoints:
        return None

    checkpoints.sort(key=lambda x: x[0])
    return checkpoints[-1][1]

latest_checkpoint = get_latest_checkpoint(cfg.OUTPUT_DIR)
print("Latest checkpoint:", latest_checkpoint)

Latest checkpoint: /content/drive/MyDrive/Data298ModelTraining/checkpoints/deepseek_rca_7b/checkpoint-1400


In [24]:
from peft import PeftModel, LoraConfig, prepare_model_for_kbit_training, get_peft_model
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for this training setup.")

if torch.cuda.is_bf16_supported():
    compute_dtype = torch.bfloat16
    use_bf16 = True
    use_fp16 = False
else:
    compute_dtype = torch.float16
    use_bf16 = False
    use_fp16 = True

bnb_config = BitsAndBytesConfig(
    load_in_4bit=cfg.LOAD_IN_4BIT,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

base_model = AutoModelForCausalLM.from_pretrained(
    cfg.MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model)

if latest_checkpoint is not None:
    model = PeftModel.from_pretrained(
        base_model,
        latest_checkpoint,
        is_trainable=True,
    )
    print("Loaded model weights from checkpoint:", latest_checkpoint)
else:
    peft_config = LoraConfig(
        r=cfg.LORA_R,
        lora_alpha=cfg.LORA_ALPHA,
        lora_dropout=cfg.LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "up_proj", "down_proj", "gate_proj"
        ],
    )
    model = get_peft_model(base_model, peft_config)
    print("No checkpoint found. Starting fresh.")

model.gradient_checkpointing_enable()
model.print_trainable_parameters()

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

Loaded model weights from checkpoint: /content/drive/MyDrive/Data298ModelTraining/checkpoints/deepseek_rca_7b/checkpoint-1400
trainable params: 37,478,400 || all params: 6,947,844,096 || trainable%: 0.5394


In [25]:
cfg.OUTPUT_DIR = cfg.OUTPUT_DIR + "_restart_run"
print("New output dir:", cfg.OUTPUT_DIR)

New output dir: /content/drive/MyDrive/Data298ModelTraining/checkpoints/deepseek_rca_7b_restart_run


In [29]:
training_args = TrainingArguments(
    output_dir=cfg.OUTPUT_DIR,
    num_train_epochs=cfg.EPOCHS,
    per_device_train_batch_size=cfg.PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=cfg.PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=cfg.GRADIENT_ACCUMULATION_STEPS,
    learning_rate=cfg.LEARNING_RATE,
    weight_decay=cfg.WEIGHT_DECAY,
    warmup_ratio=cfg.WARMUP_RATIO,
    logging_steps=10,
    save_steps=100,
    eval_steps=100,
    eval_strategy="steps",
    save_strategy="steps",
    save_total_limit=3,
    bf16=use_bf16,
    fp16=use_fp16,
    report_to="none",
    lr_scheduler_type="cosine",
    seed=cfg.SEED,
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    optim="adamw_torch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    train_sampling_strategy="group_by_length",
    length_column_name="length",
)

print(training_args)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=False,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=100,
eval_strategy=IntervalStrategy.STEPS,
eval_use_gather_object=False

In [30]:
import os

print("Output dir:", training_args.output_dir)
print(os.listdir(training_args.output_dir))

Output dir: /content/drive/MyDrive/Data298ModelTraining/checkpoints/deepseek_rca_7b_restart_run
[]


In [32]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("Trainer created successfully")

Trainer created successfully


In [33]:
train_result = trainer.train()

trainer.save_model(training_args.output_dir)
tokenizer.save_pretrained(training_args.output_dir)

print(train_result)

Step,Training Loss,Validation Loss
100,0.042368,0.194226
200,0.006668,0.056768
300,0.002251,0.028201
400,0.000929,0.005045
500,0.000216,0.000251
600,0.000118,0.001203
700,0.000084,0.000085
800,0.000059,0.000060
900,0.000049,0.000050
1000,0.000041,0.000042


TrainOutput(global_step=2100, training_loss=0.004503152076019129, metrics={'train_runtime': 46238.0656, 'train_samples_per_second': 0.363, 'train_steps_per_second': 0.045, 'total_flos': 7.798233508538941e+17, 'train_loss': 0.004503152076019129, 'epoch': 3.0})


Evaluation


In [38]:
import torch
import pandas as pd
from collections import defaultdict

# ===== 1. Load trained model =====
from peft import PeftModel
from transformers import AutoModelForCausalLM

base = AutoModelForCausalLM.from_pretrained(
    cfg.MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base, training_args.output_dir)
model.eval()

print("Model loaded for evaluation")


# ===== 2. Generation function =====
def generate_rca(sample, max_new_tokens=160):
    prompt = build_prompt(sample)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=cfg.MAX_SEQ_LEN,
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            top_p=1.0,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    pred = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    return pred


# ===== 3. Scenario classifier =====
def predict_scenario(text):
    text = text.lower()

    rules = {
        "oomkilled_limit_too_low": ["oom", "memory limit"],
        "crashloop_bad_args": ["crashloop", "argument", "args"],
        "imagepull_bad_tag": ["image tag", "manifest unknown"],
        "imagepull_registry_auth": ["authentication", "credentials"],
        "pvc_not_found_mountfail": ["pvc", "persistentvolumeclaim"],
        "failedscheduling_taint": ["taint", "toleration"],
        "failedscheduling_insufficient_memory": ["insufficient memory"],
        "failedscheduling_insufficient_cpu": ["insufficient cpu"],
        "nodeselector_mismatch": ["nodeselector", "node selector"],
        "rbac_forbidden": ["rbac", "forbidden"],
        "dns_resolution_failure": ["dns", "resolve"],
        "service_connection_refused": ["connection refused"],
        "createcontainerconfigerror_missing_secret": ["secret"],
        "createcontainerconfigerror_bad_configmap_key": ["configmap"],
    }

    for scenario, kws in rules.items():
        if any(k in text for k in kws):
            return scenario

    return "unknown"


# ===== 4. Run evaluation =====
num_samples = min(700, len(test_rows))
print("Evaluating on:", num_samples)

correct = 0
results = []

for i in range(num_samples):
    sample = test_rows[i]

    pred_text = generate_rca(sample)
    pred_scenario = predict_scenario(pred_text)
    true_scenario = sample["scenario_id"]

    is_correct = int(pred_scenario == true_scenario)
    correct += is_correct

    results.append({
        "true": true_scenario,
        "pred": pred_scenario,
        "correct": is_correct,
        "text": pred_text,
    })

    if (i+1) % 50 == 0:
        print(f"{i+1}/{num_samples} | acc={correct/(i+1):.4f}")


accuracy = correct / num_samples
print("\nFINAL TEST ACCURACY:", accuracy)


# ===== 5. Save results =====
df = pd.DataFrame(results)

save_path = training_args.output_dir + "/test_results.csv"
df.to_csv(save_path, index=False)

print("Saved results to:", save_path)


# ===== 6. Per-scenario accuracy =====
stats = defaultdict(lambda: {"c":0,"t":0})

for r in results:
    stats[r["true"]]["t"] += 1
    if r["correct"]:
        stats[r["true"]]["c"] += 1

print("\nPer-scenario accuracy:")
for k in sorted(stats.keys()):
    c = stats[k]["c"]
    t = stats[k]["t"]
    print(f"{k:40s} {c}/{t} = {c/t:.3f}")


# ===== 7. Show mistakes =====
wrong = df[df["correct"] == 0].head(5)

print("\nSample mistakes:\n")

for _, row in wrong.iterrows():
    print("="*80)
    print("TRUE:", row["true"])
    print("PRED:", row["pred"])
    print("\nOUTPUT:\n", row["text"])

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Model loaded for evaluation
Evaluating on: 700
50/700 | acc=0.7200
100/700 | acc=0.7300
150/700 | acc=0.7333
200/700 | acc=0.7050
250/700 | acc=0.6840
300/700 | acc=0.6933
350/700 | acc=0.7029
400/700 | acc=0.7025
450/700 | acc=0.7022
500/700 | acc=0.7000
550/700 | acc=0.7000
600/700 | acc=0.7033
650/700 | acc=0.7108
700/700 | acc=0.7143

FINAL TEST ACCURACY: 0.7142857142857143
Saved results to: /content/drive/MyDrive/Data298ModelTraining/checkpoints/deepseek_rca_7b_restart_run/test_results.csv

Per-scenario accuracy:
crashloop_bad_args                       50/50 = 1.000
createcontainerconfigerror_bad_configmap_key 50/50 = 1.000
createcontainerconfigerror_missing_secret 50/50 = 1.000
dns_resolution_failure                   50/50 = 1.000
failedscheduling_insufficient_cpu        0/50 = 0.000
failedscheduling_insufficient_memory     0/50 = 0.000
failedscheduling_taint                   50/50 = 1.000
imagepull_bad_tag                        0/50 = 0.000
imagepull_registry_auth           